# hermit-llm — Playground Notebook

Use this notebook to explore and experiment with each component interactively.
Run cells one by one as you implement each module.

**Sections:**
1. CharTokenizer
2. CausalSelfAttention
3. FeedForward
4. TransformerBlock
5. MiniGPT — forward pass
6. Training loop
7. Text generation
8. Attention visualization

In [ ]:
# Make sure hermit-llm root is on the path
import sys, os
sys.path.insert(0, os.path.abspath('.'))

## 1. CharTokenizer

In [ ]:
from tokenizer import CharTokenizer

corpus = "hello world! this is hermit-ai."
tok = CharTokenizer(corpus)

print(f"Vocabulary size: {tok.vocab_size}")
print(f"Characters: {sorted(tok._char_to_id.keys())}")

sample = "hello"
encoded = tok.encode(sample)
decoded = tok.decode(encoded)
print(f"\nencode('{sample}') -> {encoded}")
print(f"decode({encoded}) -> '{decoded}'")
print(f"Round-trip OK: {sample == decoded}")

## 2. CausalSelfAttention

In [ ]:
import torch
from attention import CausalSelfAttention

B, T, C = 2, 8, 64   # batch=2, seq_len=8, embed_dim=64
attn = CausalSelfAttention(embed_dim=C, num_heads=4, dropout=0.0)
attn.eval()

x = torch.randn(B, T, C)
out = attn(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Shape preserved: {x.shape == out.shape}")

## 3. FeedForward

In [ ]:
from feedforward import FeedForward

ff = FeedForward(embed_dim=64, dropout=0.0)
ff.eval()

x = torch.randn(2, 8, 64)
out = ff(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")

## 4. TransformerBlock

In [ ]:
from block import TransformerBlock

block = TransformerBlock(embed_dim=64, num_heads=4, dropout=0.0)
block.eval()

x = torch.randn(2, 8, 64)
out = block(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")

## 5. MiniGPT — forward pass

In [ ]:
from model import MiniGPT

model = MiniGPT(
    vocab_size=tok.vocab_size,
    embed_dim=64,
    num_heads=4,
    num_layers=2,
    context_length=32,
    dropout=0.0,
)
model.eval()

# Random token sequence
idx = torch.randint(0, tok.vocab_size, (1, 16))  # batch=1, seq_len=16
logits, loss = model(idx)
print(f"Logits shape: {logits.shape}")  # (1, 16, vocab_size)
print(f"Loss (no targets): {loss}")

# With targets
targets = torch.randint(0, tok.vocab_size, (1, 16))
logits, loss = model(idx, targets)
print(f"Loss (with targets): {loss.item():.4f}")

## 6. Training loop (short run)

In [ ]:
from train import HyperParams, train

hp = HyperParams(
    max_steps=200,
    eval_interval=50,
    embed_dim=64,
    num_heads=4,
    num_layers=2,
    context_length=64,
    batch_size=16,
    checkpoint_path="checkpoint_notebook.pt",
)

train(hp)

## 7. Text generation

In [ ]:
from generate import generate_text

prompt = "Once upon"
result = generate_text(
    prompt=prompt,
    checkpoint_path="checkpoint_notebook.pt",
    max_new_tokens=200,
    temperature=0.8,
)
print(result)

## 8. Attention visualization

Visualize the attention weights for a given input sequence.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import torch
import torch.nn.functional as F
import math
from generate import load_checkpoint

model, tokenizer, _ = load_checkpoint("checkpoint_notebook.pt")
model.eval()

prompt = "hello world"
ids = tokenizer.encode(prompt)
idx = torch.tensor([ids])

# Hook to capture attention weights from the first block
attention_weights = {}

def hook_fn(module, input, output):
    # Re-compute attention weights for visualization
    x = input[0]
    B, T, C = x.shape
    qkv = module.qkv_proj(x)
    Q, K, V = qkv.split(C, dim=2)
    H = module.num_heads
    D = C // H
    Q = Q.view(B, T, H, D).transpose(1, 2)
    K = K.view(B, T, H, D).transpose(1, 2)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
    mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
    scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    attention_weights['block0'] = weights.detach()

handle = model.blocks[0].attention.register_forward_hook(hook_fn)

with torch.no_grad():
    model(idx)

handle.remove()

# Plot attention weights for head 0
w = attention_weights['block0'][0, 0].numpy()  # (T, T)
chars = list(prompt)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(w, cmap='Blues')
ax.set_xticks(range(len(chars)))
ax.set_yticks(range(len(chars)))
ax.set_xticklabels(chars)
ax.set_yticklabels(chars)
ax.set_title('Attention weights — block 0, head 0')
plt.colorbar(im)
plt.tight_layout()
plt.show()